In [1]:
import io
import soundfile as sf
from datasets import load_dataset, Audio

ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
ds = ds.cast_column("audio", Audio(decode=False))

clips = []
for i in range(8):
    row = ds[i]
    wave, rate = sf.read(io.BytesIO(row["audio"]["bytes"]))
    clips.append({"audio": wave, "rate": rate, "ref": row["text"]})

print(len(ds), "clips in the dataset; using", len(clips))
for c in clips[:3]:
    print(c["rate"], "Hz |", round(len(c["audio"]) / c["rate"], 1), "s |", c["ref"])

README.md:   0%|          | 0.00/520 [00:00<?, ?B/s]

clean/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 9.19MB            

clean/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/73 [00:00<?, ? examples/s]

73 clips in the dataset; using 8
16000 Hz | 5.9 s | MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL
16000 Hz | 4.8 s | NOR IS MISTER QUILTER'S MANNER LESS INTERESTING THAN HIS MATTER
16000 Hz | 12.5 s | HE TELLS US THAT AT THIS FESTIVE SEASON OF THE YEAR WITH CHRISTMAS AND ROAST BEEF LOOMING BEFORE US SIMILES DRAWN FROM EATING AND ITS RESULTS OCCUR MOST READILY TO THE MIND


In [2]:
import re
import torch
!pip install -q jiwer
import jiwer
from transformers import WhisperProcessor, WhisperForConditionalGeneration

name = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(name)
model = WhisperForConditionalGeneration.from_pretrained(name).to("cuda")

def normalize(s):
    s = s.lower()
    s = re.sub(r"[^a-z0-9' ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

inputs = processor(
    [c["audio"] for c in clips],
    sampling_rate=16000,
    return_tensors="pt",
).to("cuda")

with torch.no_grad():
    ids = model.generate(inputs.input_features, language="en", task="transcribe")

hyps = processor.batch_decode(ids, skip_special_tokens=True)

refs_n = [normalize(c["ref"]) for c in clips]
hyps_n = [normalize(h) for h in hyps]

print("sanity check (should be 0.0):", jiwer.wer(refs_n, refs_n))
print("overall WER:", round(jiwer.wer(refs_n, hyps_n), 3))
print()
for r, h in zip(refs_n, hyps_n):
    print(round(jiwer.wer(r, h), 3), "|", r)
    print("      ", h)

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

... (5 outputs left out)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask is not set with a batched input, and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


sanity check (should be 0.0): 0.0
overall WER: 0.1

0.059 | mister quilter is the apostle of the middle classes and we are glad to welcome his gospel
       mr quilter is the apostle of the middle classes and we are glad to welcome his gospel
0.1 | nor is mister quilter's manner less interesting than his matter
       nor is mr quilter's manner less interesting than his matter
0.031 | he tells us that at this festive season of the year with christmas and roast beef looming before us similes drawn from eating and its results occur most readily to the mind
       he tells us that at this festive season of the year with christmas and roast beef looming before us symbolies drawn from eating and its results occur most readily to the mind
0.042 | he has grave doubts whether sir frederick leighton's work is really greek after all and can discover in it but little of rocky ithaca
       he has grave doubts whether sir frederick layton's work is really greek after all and can discover in it but

In [3]:
def normalize2(s):
    s = normalize(s)
    s = re.sub(r"\bmister\b", "mr", s)
    return s

refs_n2 = [normalize2(c["ref"]) for c in clips]
hyps_n2 = [normalize2(h) for h in hyps]

out = jiwer.process_words(refs_n2, hyps_n2)

print("sanity check (should be 0.0):", jiwer.wer(refs_n2, refs_n2))
print("overall WER:", round(out.wer, 3))
print("correct words:", out.hits)
print("substitutions:", out.substitutions)
print("deletions:", out.deletions)
print("insertions:", out.insertions)

sanity check (should be 0.0): 0.0
overall WER: 0.07
correct words: 186
substitutions: 12
deletions: 2
insertions: 0


In [4]:
import os
import librosa

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/vfxbymani/tamil-clip/Pachaimamalai (mp3cut.net).mp3


In [5]:
import librosa
import torch
from IPython.display import Audio

path = "/kaggle/input/datasets/vfxbymani/tamil-clip/Pachaimamalai (mp3cut.net).mp3"

audio_ta, _ = librosa.load(path, sr=16000)
duration = len(audio_ta) / 16000
print("duration (seconds):", round(duration, 1))

audio_ta = audio_ta[: 30 * 16000]

inputs = processor(audio_ta, sampling_rate=16000, return_tensors="pt").to("cuda")

with torch.no_grad():
    ids = model.generate(inputs.input_features, language="ta", task="transcribe")

tamil_text = processor.batch_decode(ids, skip_special_tokens=True)[0]
print(tamil_text)

Audio(audio_ta, rate=16000)

duration (seconds): 21.6
 பத்ஷை மாமலை போல் மேனி பவழ வாய்க்கமல செங்கம்


In [6]:
import re
import jiwer

reference = """பச்சைமா மலைபோல் மேனி
பவளவாய் கமலச் செங்கண்"""

def norm_ta(s):
    s = re.sub(r"[^\u0B80-\u0BFF ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

ref_n = norm_ta(reference)
hyp_n = norm_ta(tamil_text)

print("reference:", ref_n)
print("model:    ", hyp_n)
print()
print("sanity check (should be 0.0):", jiwer.wer(ref_n, ref_n), jiwer.cer(ref_n, ref_n))
print("WER:", round(jiwer.wer(ref_n, hyp_n), 3))
print("CER:", round(jiwer.cer(ref_n, hyp_n), 3))

ref_ns = ref_n.replace(" ", "")
hyp_ns = hyp_n.replace(" ", "")
print("CER ignoring spaces:", round(jiwer.cer(ref_ns, hyp_ns), 3))
print("reference words:", len(ref_n.split()), "| model words:", len(hyp_n.split()))

reference: பச்சைமா மலைபோல் மேனி பவளவாய் கமலச் செங்கண்
model:     பத்ஷை மாமலை போல் மேனி பவழ வாய்க்கமல செங்கம்

sanity check (should be 0.0): 0.0 0.0
WER: 1.0
CER: 0.286
CER ignoring spaces: 0.216
reference words: 6 | model words: 7


In [7]:
import gc
import torch
import jiwer
from transformers import WhisperProcessor, WhisperForConditionalGeneration

tamil_small = tamil_text

del model
gc.collect()
torch.cuda.empty_cache()

name = "openai/whisper-medium"
processor_m = WhisperProcessor.from_pretrained(name)
model_m = WhisperForConditionalGeneration.from_pretrained(name, dtype=torch.float16).to("cuda")

inputs = processor_m(audio_ta, sampling_rate=16000, return_tensors="pt")
feats = inputs.input_features.to("cuda", dtype=torch.float16)

with torch.no_grad():
    ids = model_m.generate(feats, language="ta", task="transcribe")

tamil_medium = processor_m.batch_decode(ids, skip_special_tokens=True)[0]

for label, text in [("small ", tamil_small), ("medium", tamil_medium)]:
    h = norm_ta(text)
    print(label, "| WER", round(jiwer.wer(ref_n, h), 3),
          "| CER", round(jiwer.cer(ref_n, h), 3),
          "| CER no spaces", round(jiwer.cer(ref_n.replace(" ", ""), h.replace(" ", "")), 3))
    print("       ", h)

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.99k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

... (4 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.75k [00:00<?, ?B/s]

small  | WER 1.0 | CER 0.286 | CER no spaces 0.216
        பத்ஷை மாமலை போல் மேனி பவழ வாய்க்கமல செங்கம்
medium | WER 1.0 | CER 0.19 | CER no spaces 0.108
        பத்சை மாமலை போல் மே நீ பவளவாய் கமலச் செங்கம்


In [8]:
!pip install -q rapidfuzz
from rapidfuzz import fuzz

reference_lines = [
    "பச்சை மாமலைபோல் மேனி",
    "பவள வாய் கமலச் செங்கண்",
]

def squash(s):
    return norm_ta(s).replace(" ", "")

ref_joined = " ".join(reference_lines)
score = fuzz.ratio(squash(tamil_medium), squash(ref_joined))
print("similarity score (0-100):", round(score, 1))

THRESHOLD = 75
if score >= THRESHOLD:
    final_lines = reference_lines
    print("auto-corrected to known text")
else:
    final_lines = [tamil_medium.strip()]
    print("FLAG FOR HUMAN REVIEW: too different from the known text")

def fmt_time(t):
    ms = int(round(t * 1000))
    h, ms = divmod(ms, 3_600_000)
    m, ms = divmod(ms, 60_000)
    s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

end = len(audio_ta) / 16000
srt = f"1\n{fmt_time(0)} --> {fmt_time(end)}\n" + "\n".join(final_lines) + "\n"
print()
print(srt)

with open("tamil_clip.srt", "w", encoding="utf-8") as f:
    f.write(srt)

similarity score (0-100): 89.2
auto-corrected to known text

1
00:00:00,000 --> 00:00:21,565
பச்சை மாமலைபோல் மேனி
பவள வாய் கமலச் செங்கண்

